# AccelForge Rack `!Network` — Matmul on a 4x4 Rack

This notebook adapts the network-internals analysis to AccelForge's **rack** example.

The rack architecture models a multi-node system:

```
MainMemory          (shared pool, all tensors)
  NoC               (!Network — inter-node fabric)
    HBM             (per-node, spatial 4x4 = 16 nodes)
      Buffer        (per-node scratchpad)
        MAC         (compute)
```

**Key difference from the `hierarchical` example:**
- Hierarchical has TWO networks (PeArray + MacArray) with 2x2 spatial each
- Rack has ONE network (NoC) with 4x4 spatial = 16 nodes
- The rack is structurally similar to your Passage EVK100 model!

**Original rack example has `actions: []` on NoC (no cost).** We test
three variants:
1. Original (zero cost) — baseline
2. Electrical mesh (10 pJ/bit per hop)
3. Photonic crossbar (1.9 pJ/bit flat, via hops=0 + transceiver trick)

## 1. Setup

In [ ]:
import accelforge as af
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import tempfile, os

RACK_ARCH = af.examples.arches.networked.rack
WORKLOAD = af.examples.workloads.matmuls

print("Rack architecture:", RACK_ARCH)
print("Workload:         ", WORKLOAD)

## 2. Print the rack architecture

In [ ]:
print(Path(RACK_ARCH).read_text())

## 3. Run the original rack (NoC with empty actions)

Since `actions: []`, the NoC defines **no hop action at all**. AccelForge has
nothing to count or charge. This is a skeleton — the library ships it as a
structural template, not a runnable benchmark. No test or example in
AccelForge ever actually runs the rack with a workload.

We run it here just to confirm: zero energy, zero hops.

In [ ]:
# Use same matmul dimensions as notebook 6 for comparison
M = 8
KN = 8

spec_rack = af.Spec.from_yaml(
    WORKLOAD,
    RACK_ARCH,
    jinja_parse_data={"N_EINSUMS": 1, "M": M, "KN": KN},
)
spec_rack.mapper.metrics = af.mapper.Metrics.ENERGY
mappings_rack = spec_rack.map_workload_to_arch(
    print_progress=False, print_number_of_pmappings=False
)

if len(mappings_rack) > 0:
    best_rack = mappings_rack[0]
    print(f"Energy:  {float(best_rack.energy())} units")
    print(f"Latency: {float(best_rack.latency())} units")
    print()
    print("As expected: zero energy (all actions have energy=0)")
else:
    print("No valid mappings found!")

## 4. Create modified rack architectures with real costs

We write three YAML variants to a temp directory:
1. **Rack + electrical mesh** — NoC has hops energy = 10 pJ/bit
2. **Rack + photonic (per-hop)** — NoC has hops energy = 1.9 pJ/bit  
3. **Rack + photonic (flat, like your Passage model)** — NoC hops=0, plus a Transceiver memory

In [ ]:
TMPDIR = Path(tempfile.mkdtemp())

# --- Variant A: Electrical mesh (10 pJ/bit per hop) ---
RACK_ELECTRICAL = TMPDIR / "rack_electrical.yaml"
RACK_ELECTRICAL.write_text("""
arch:
  nodes:
  - !Memory
    name: MainMemory
    size: inf
    area: 0
    leak_power: 0
    tensors: {keep: All}
    actions:
    - {name: read, energy: 0, latency: 0}
    - {name: write, energy: 0, latency: 0}

  - !Network
    name: NoC
    area: 0
    leak_power: 0
    actions:
    - {name: hops, energy: 10e-12, latency: 5e-9}

  - !Memory
    name: HBM
    size: inf
    area: 0
    leak_power: 0
    tensors: {keep: ~MainMemory, may_keep: All}
    actions:
    - {name: read, energy: 0, latency: 0}
    - {name: write, energy: 0, latency: 0}
    spatial:
    - {name: X, fanout: 4}
    - {name: Y, fanout: 4}

  - !Memory
    name: Buffer
    size: inf
    area: 0
    leak_power: 0
    actions:
    - {name: read, energy: 0, latency: 0}
    - {name: write, energy: 0, latency: 0}

  - !Compute
    name: MAC
    area: 0
    leak_power: 0
    actions:
    - {name: compute, energy: 0, latency: 0}
""")

# --- Variant B: Photonic per-hop (1.9 pJ/bit, single hop = crossbar) ---
RACK_PHOTONIC_HOP = TMPDIR / "rack_photonic_hop.yaml"
RACK_PHOTONIC_HOP.write_text("""
arch:
  nodes:
  - !Memory
    name: MainMemory
    size: inf
    area: 0
    leak_power: 0
    tensors: {keep: All}
    actions:
    - {name: read, energy: 0, latency: 0}
    - {name: write, energy: 0, latency: 0}

  - !Network
    name: NoC
    area: 0
    leak_power: 0
    actions:
    - {name: hops, energy: 1.9e-12, latency: 0.39e-12}

  - !Memory
    name: HBM
    size: inf
    area: 0
    leak_power: 0
    tensors: {keep: ~MainMemory, may_keep: All}
    actions:
    - {name: read, energy: 0, latency: 0}
    - {name: write, energy: 0, latency: 0}
    spatial:
    - {name: X, fanout: 4}
    - {name: Y, fanout: 4}

  - !Memory
    name: Buffer
    size: inf
    area: 0
    leak_power: 0
    actions:
    - {name: read, energy: 0, latency: 0}
    - {name: write, energy: 0, latency: 0}

  - !Compute
    name: MAC
    area: 0
    leak_power: 0
    actions:
    - {name: compute, energy: 0, latency: 0}
""")

# --- Variant C: Photonic flat (your Passage model: hops=0, transceiver toll) ---
RACK_PHOTONIC_FLAT = TMPDIR / "rack_photonic_flat.yaml"
RACK_PHOTONIC_FLAT.write_text("""
arch:
  nodes:
  - !Memory
    name: MainMemory
    size: inf
    area: 0
    leak_power: 0
    tensors: {keep: All}
    actions:
    - {name: read, energy: 0, latency: 0}
    - {name: write, energy: 0, latency: 0}

  - !Network
    name: NoC
    area: 0
    leak_power: 0
    actions:
    - {name: hops, energy: 0, latency: 0}

  - !Memory
    name: Transceiver
    size: inf
    area: 0
    leak_power: 0
    tensors: {keep: ~MainMemory, may_keep: All}
    actions:
    - {name: read, energy: 1.9e-12, latency: 0.39e-12}
    - {name: write, energy: 1.9e-12, latency: 0.39e-12}
    spatial:
    - {name: X, fanout: 4}
    - {name: Y, fanout: 4}

  - !Memory
    name: HBM
    size: inf
    area: 0
    leak_power: 0
    tensors: {keep: ~Transceiver, may_keep: All}
    actions:
    - {name: read, energy: 0, latency: 0}
    - {name: write, energy: 0, latency: 0}

  - !Memory
    name: Buffer
    size: inf
    area: 0
    leak_power: 0
    actions:
    - {name: read, energy: 0, latency: 0}
    - {name: write, energy: 0, latency: 0}

  - !Compute
    name: MAC
    area: 0
    leak_power: 0
    actions:
    - {name: compute, energy: 0, latency: 0}
""")

print("Created:")
print(f"  A) Electrical mesh: {RACK_ELECTRICAL}")
print(f"  B) Photonic hop:    {RACK_PHOTONIC_HOP}")
print(f"  C) Photonic flat:   {RACK_PHOTONIC_FLAT}")

## 5. Run all variants — small matmul (M=8, KN=8)

Same dimensions as the hierarchical notebook for direct comparison.
Only the NoC and memory energies differ.

In [ ]:
def run_rack_variant(arch_path, label, M=8, KN=8):
    """Run mapper on a rack variant, return (mapping, energy, latency, hop_details)."""
    spec = af.Spec.from_yaml(
        WORKLOAD, str(arch_path),
        jinja_parse_data={"N_EINSUMS": 1, "M": M, "KN": KN},
    )
    spec.mapper.metrics = af.mapper.Metrics.ENERGY
    mappings = spec.map_workload_to_arch(
        print_progress=False, print_number_of_pmappings=False
    )
    if len(mappings) == 0:
        print(f"  [{label}] No valid mappings!")
        return None, None, None, {}
    best = mappings[0]
    e = float(best.energy())
    l = float(best.latency())
    
    # Extract hop details
    row = best.data.iloc[0]
    hops = {}
    for col in best.data.columns:
        if "hops" in col:
            hops[col] = float(row[col])
    
    print(f"  [{label:30s}]  Energy: {e:14.6f}  |  Latency: {l:14.6f}")
    return best, e, l, hops


print("=" * 70)
print(f"  Matmul: M={M}, K={KN}, N={KN} on 4x4 rack (16 nodes)")
print("=" * 70)

results = {}
for label, arch in [
    ("Original (zero cost)", RACK_ARCH),
    ("Electrical (10 pJ/bit/hop)", RACK_ELECTRICAL),
    ("Photonic per-hop (1.9 pJ/bit)", RACK_PHOTONIC_HOP),
    ("Photonic flat (transceiver)", RACK_PHOTONIC_FLAT),
]:
    best, e, l, hops = run_rack_variant(arch, label, M=M, KN=KN)
    results[label] = {"mapping": best, "energy": e, "latency": l, "hops": hops}

## 6. Per-tensor hop breakdown for each variant

In [ ]:
for label, res in results.items():
    if res["hops"]:
        print(f"\n{label}:")
        print("-" * 60)
        for col, val in sorted(res["hops"].items()):
            parts = col.split("<SEP>")
            if len(parts) >= 4:
                network, tensor = parts[2], parts[3]
                print(f"  {network:15s} x {tensor:4s}  →  {val:12,.0f} hop-bits")
        print(f"  {'Total':15s}         →  {sum(res['hops'].values()):12,.0f} hop-bits")
    elif res["energy"] is not None:
        print(f"\n{label}: no hop columns (NoC has actions: [])")

## 7. Per-component energy breakdown

In [ ]:
for label, res in results.items():
    if res["mapping"] is not None:
        per_comp = res["mapping"].energy(per_component=True)
        nonzero = {k: float(v) for k, v in per_comp.items() if float(v) != 0}
        if nonzero:
            print(f"\n{label}:")
            for comp, e in sorted(nonzero.items(), key=lambda x: -x[1]):
                print(f"  {comp:20s}  {e:.6e}")
        else:
            print(f"\n{label}: all components have zero energy")

## 8. Scale up — realistic LLM-sized matmul

Run the same variants with a decode_mlp workload (M=1, K=4096, N=16384)
to see how the rack architecture handles real-world data volumes.

In [ ]:
# Note: the built-in matmuls.yaml uses 'KN' for both K and N (square inner dims).
# For a more realistic workload we use larger values.
# M is rows, KN is the inner/output dimension size.

llm_rows = []
for m_val, kn_val, wl_name in [
    (8, 8, "tiny (8x8x8)"),
    (8, 64, "small (8x64x64)"),
    (8, 128, "medium (8x128x128)"),
    (32, 128, "large (32x128x128)"),
]:
    print(f"\n{'='*60}")
    print(f"  Matmul: M={m_val}, KN={kn_val}")
    print(f"{'='*60}")
    for label, arch in [
        ("Electrical", RACK_ELECTRICAL),
        ("Photonic (per-hop)", RACK_PHOTONIC_HOP),
        ("Photonic (flat)", RACK_PHOTONIC_FLAT),
    ]:
        _, e, l, hops = run_rack_variant(arch, label, M=m_val, KN=kn_val)
        if e is not None:
            llm_rows.append({
                "Workload": wl_name,
                "M": m_val, "KN": kn_val,
                "MACs": m_val * kn_val * kn_val,
                "Interconnect": label,
                "Energy": e,
                "Latency": l,
                "Total hops": sum(hops.values()) if hops else 0,
            })

df_llm = pd.DataFrame(llm_rows)
display(df_llm)

## 9. Compute improvement: Photonic vs. Electrical

In [ ]:
if len(df_llm) > 0:
    imp_rows = []
    for wl in df_llm["Workload"].unique():
        sub = df_llm[df_llm["Workload"] == wl]
        e_elec = sub[sub["Interconnect"] == "Electrical"]["Energy"].values
        if len(e_elec) == 0 or e_elec[0] == 0:
            continue
        e_elec = e_elec[0]
        for ic in ["Photonic (per-hop)", "Photonic (flat)"]:
            e_ph = sub[sub["Interconnect"] == ic]["Energy"].values
            if len(e_ph) > 0 and e_ph[0] is not None:
                reduction = 100 * (1 - e_ph[0] / e_elec) if e_elec > 0 else 0
                imp_rows.append({
                    "Workload": wl,
                    "Photonic Model": ic,
                    "Elec Energy": f"{e_elec:.6e}",
                    "Phot Energy": f"{e_ph[0]:.6e}",
                    "Reduction (%)": round(reduction, 1),
                })
    if imp_rows:
        df_imp = pd.DataFrame(imp_rows)
        display(df_imp)

## 10. Unit-cost baseline — raw hop counting

The original rack has no baseline because `actions: []`. So we create one:
**1 unit of energy per hop** (same as the hierarchical example). This isolates
the pure hop-counting mechanics — how many hop-weighted bits does AccelForge
compute for a 4x4 spatial array vs. the hierarchical 2x(2x2)?

This IS the baseline. Everything above (electrical 10 pJ, photonic 1.9 pJ)
is just this number multiplied by the per-hop cost.

In [ ]:
# Rack with 1 unit/hop for direct comparison
RACK_UNIT = TMPDIR / "rack_unit.yaml"
RACK_UNIT.write_text("""
arch:
  nodes:
  - !Memory
    name: MainMemory
    size: inf
    area: 0
    leak_power: 0
    tensors: {keep: All}
    actions:
    - {name: read, energy: 0, latency: 0}
    - {name: write, energy: 0, latency: 0}

  - !Network
    name: NoC
    area: 0
    leak_power: 0
    actions:
    - {name: hops, energy: 1, latency: 0}

  - !Memory
    name: HBM
    size: inf
    area: 0
    leak_power: 0
    tensors: {keep: ~MainMemory, may_keep: All}
    actions:
    - {name: read, energy: 0, latency: 0}
    - {name: write, energy: 0, latency: 0}
    spatial:
    - {name: X, fanout: 4}
    - {name: Y, fanout: 4}

  - !Memory
    name: Buffer
    size: inf
    area: 0
    leak_power: 0
    actions:
    - {name: read, energy: 0, latency: 0}
    - {name: write, energy: 0, latency: 0}

  - !Compute
    name: MAC
    area: 0
    leak_power: 0
    actions:
    - {name: compute, energy: 0, latency: 0}
""")

# Run rack with unit cost
_, e_rack_unit, _, hops_rack = run_rack_variant(RACK_UNIT, "Rack (1/hop)", M=8, KN=8)

# Run hierarchical with unit cost (built-in already has energy=1)
spec_hier = af.Spec.from_yaml(
    WORKLOAD,
    af.examples.arches.networked.hierarchical,
    jinja_parse_data={"N_EINSUMS": 1, "M": 8, "KN": 8},
)
spec_hier.mapper.metrics = af.mapper.Metrics.ENERGY
mappings_hier = spec_hier.map_workload_to_arch(
    print_progress=False, print_number_of_pmappings=False
)
best_hier = mappings_hier[0]
e_hier = float(best_hier.energy())

print(f"\nComparison (M=8, KN=8, energy=1/hop):")
print(f"  Hierarchical (2x PeArray+MacArray, 2x2 spatial each): {e_hier:,.0f} total hop-bits")
print(f"  Rack (1x NoC, 4x4 spatial):                          {e_rack_unit:,.0f} total hop-bits")
if e_hier > 0:
    print(f"  Ratio (rack/hierarchical): {e_rack_unit/e_hier:.2f}x")

In [ ]:
# Detailed per-tensor comparison
print("\nRack — per-tensor hops:")
for col, val in sorted(hops_rack.items()):
    parts = col.split("<SEP>")
    if len(parts) >= 4:
        print(f"  {parts[2]:12s} x {parts[3]:4s} = {val:8,.0f}")

print("\nHierarchical — per-tensor hops:")
hier_row = best_hier.data.iloc[0]
for col in sorted(best_hier.data.columns):
    if "hops" in col:
        parts = col.split("<SEP>")
        if len(parts) >= 4:
            print(f"  {parts[2]:12s} x {parts[3]:4s} = {float(hier_row[col]):8,.0f}")

## 11. Key takeaways

1. **Rack architecture** = single `!Network` + single 4x4 spatial level — structurally identical to your Passage model
2. **Original rack has `actions: []`** — no hop cost, so zero energy. You must add actions to see network effects.
3. **Per-hop model** (variants A/B): AccelForge charges `energy_per_hop × (fanout-1) × volume` for multicast,
   `0.5 × N × (N-1) × volume` for unicast — same formulas regardless of architecture.
4. **Photonic flat model** (variant C): sets hops=0, adds a Transceiver `!Memory` with flat per-bit cost.
   This charges **every read/write through the transceiver** uniformly — no distance dependence.
5. **Rack vs. hierarchical**: larger spatial fanout (4x4 vs 2x2) means more hops per multicast,
   so the single-network rack may have different total hops than the two-network hierarchical.